In [15]:

# %% [imports]
import os
import glob
import json
import pickle
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Any, Union
from dataclasses import dataclass, asdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import MiniBatchKMeans, KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report,
    f1_score,
    precision_score,
    recall_score
)
#%%
@dataclass
class ExperimentConfig:
    """Configuration for a single experiment"""
    # Data parameters
    data_dir: str = "flowers"
    image_max_size: int = 400
    test_size: float = 0.2
    random_state: int = 42
    
    # Feature extraction parameters
    sift_n_features: int = 0  # 0 = unlimited
    sift_contrast_threshold: float = 0.04
    
    # Vocabulary parameters
    vocab_size: int = 150
    use_minibatch_kmeans: bool = True
    kmeans_batch_size: int = 1000
    
    # Classifier parameters
    classifier_type: str = 'knn'  # 'knn' or 'svm'
    
    # KNN-specific parameters
    knn_neighbors: int = 5
    knn_metric: str = 'euclidean'
    knn_weights: str = 'uniform'  # 'uniform' or 'distance'
    
    # SVM-specific parameters
    svm_kernel: str = 'rbf'  # 'linear', 'poly', 'rbf', 'sigmoid'
    svm_C: float = 1.0
    svm_gamma: str = 'scale'  # 'scale', 'auto', or float value
    svm_degree: int = 3  # for polynomial kernel
    
    # Feature normalization
    normalize_histograms: bool = True
    use_feature_scaling: bool = False  # StandardScaler before classification
    
    # Experiment metadata
    experiment_name: str = "baseline"
    experiment_description: str = ""
    
    def to_dict(self) -> Dict:
        return asdict(self)


In [16]:
# %% [data_loading]
class DataLoader:
    """Handles loading and preprocessing of image data"""
    
    def __init__(self, config: ExperimentConfig):
        self.config = config
        
    def load_images_and_labels(self) -> Tuple[List[np.ndarray], np.ndarray, List[str]]:
        """
        Load images from directory structure: data_dir/<class>/*.jpg
        
        Returns:
            images: List of BGR images (resized)
            labels: Array of integer labels
            class_names: List of class names
        """
        images = []
        labels = []
        
        class_names = sorted(os.listdir(self.config.data_dir))
        class_to_idx = {c: i for i, c in enumerate(class_names)}
        
        print(f"Loading images from: {self.config.data_dir}")
        print(f"Found classes: {class_names}")
        
        for class_name in class_names:
            class_dir = os.path.join(self.config.data_dir, class_name)
            image_files = glob.glob(os.path.join(class_dir, "*"))
            
            for img_path in image_files:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Could not read {img_path}")
                    continue
                
                # Resize image to limit computation
                img_resized = self._resize_image(img)
                images.append(img_resized)
                labels.append(class_to_idx[class_name])
        
        print(f"Loaded {len(images)} images from {len(class_names)} classes")
        return images, np.array(labels), class_names
    
    def _resize_image(self, img: np.ndarray) -> np.ndarray:
        """Resize image maintaining aspect ratio"""
        h, w = img.shape[:2]
        max_dim = max(h, w)
        
        if max_dim > self.config.image_max_size:
            scale = self.config.image_max_size / max_dim
            new_w, new_h = int(w * scale), int(h * scale)
            img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        
        return img

# %% [feature_extraction]
class FeatureExtractor:
    """Extracts SIFT features from images"""
    
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.sift = cv2.SIFT_create(
            nfeatures=config.sift_n_features,
            contrastThreshold=config.sift_contrast_threshold
        )
    
    def extract_descriptors(self, images: List[np.ndarray]) -> Tuple[List[np.ndarray], np.ndarray]:
        """
        Extract SIFT descriptors from all images
        
        Returns:
            image_descriptors: List of descriptor arrays per image (N_i x 128)
            all_descriptors_stacked: All descriptors concatenated (M x 128)
                where N_i = number of keypoints in image i
                      M = total keypoints across all images
        """
        # import it from npy array else calciulate it
        all_descriptors = None
        try:
            all_descriptors = np.load('all_descriptors.npy')
            image_descriptors = np.load('image_descriptors.npy', allow_pickle=True)
            print("Loaded all descriptors from 'all_descriptors.npy'")
        except FileNotFoundError:
            print("File 'all_descriptors.npy' not found. Extracting descriptors...")
            all_descriptors = None 
            image_descriptors = None

        if all_descriptors is not  None and image_descriptors is not None:
            return image_descriptors, all_descriptors
        image_descriptors = []
        all_descriptors = []
        
        print("Extracting SIFT descriptors...")
        for i, img in enumerate(images):
            if (i + 1) % 100 == 0:
                print(f"  Processed {i + 1}/{len(images)} images")
            
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            keypoints, descriptors = self.sift.detectAndCompute(gray, None)
            
            # Handle images with no keypoints
            if descriptors is None:
                descriptors = np.zeros((0, 128), dtype=np.float32)
            
            image_descriptors.append(descriptors)
            
            if descriptors.shape[0] > 0:
                all_descriptors.append(descriptors)
        
        if len(all_descriptors) == 0:
            raise RuntimeError("No descriptors found in any image")
        
        all_descriptors_stacked = np.vstack(all_descriptors).astype(np.float32)
        print(f"Total descriptors extracted: {all_descriptors_stacked.shape[0]}")
        
        # Save descriptors for future use
        np.save('all_descriptors.npy', all_descriptors_stacked)
        np.save('image_descriptors.npy', np.array(image_descriptors, dtype=object))
        print("Saved descriptors to 'all_descriptors.npy' and 'image_descriptors.npy'")
        return image_descriptors, all_descriptors_stacked

# %% [vocabulary_builder]
class VocabularyBuilder:
    """Builds visual vocabulary using K-Means clustering"""
    
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.kmeans = None
        self.vocabulary = None
    
    def build_vocabulary(self, descriptors: np.ndarray) -> np.ndarray:
        """
        Build visual vocabulary by clustering descriptors
        
        Args:
            descriptors: All SIFT descriptors (M x 128)
        
        Returns:
            vocabulary: Cluster centers (vocab_size x 128)
        """
        print(f"Building vocabulary with {self.config.vocab_size} visual words...")
        
        if self.config.use_minibatch_kmeans:
            self.kmeans = MiniBatchKMeans(
                n_clusters=self.config.vocab_size,
                random_state=self.config.random_state,
                batch_size=self.config.kmeans_batch_size,
                verbose=0
            )
        else:
            self.kmeans = KMeans(
                n_clusters=self.config.vocab_size,
                random_state=self.config.random_state,
                verbose=0
            )
        
        self.kmeans.fit(descriptors)
        self.vocabulary = self.kmeans.cluster_centers_
        
        print(f"Vocabulary built. Shape: {self.vocabulary.shape}")
        return self.vocabulary
    
    def compute_histograms(self, image_descriptors: List[np.ndarray]) -> np.ndarray:
        """
        Compute histogram of visual words for each image
        
        Args:
            image_descriptors: List of descriptor arrays per image
        
        Returns:
            histograms: Array of shape (n_images, vocab_size)
        """
        n_images = len(image_descriptors)
        histograms = np.zeros((n_images, self.config.vocab_size), dtype=np.float32)
        
        print("Computing BoVW histograms...")
        for i, descriptors in enumerate(image_descriptors):
            if descriptors.shape[0] == 0:
                # No features detected in this image
                continue
            
            # Assign each descriptor to nearest visual word
            visual_words = self.kmeans.predict(descriptors)
            
            # Create histogram
            hist, _ = np.histogram(
                visual_words,
                bins=np.arange(self.config.vocab_size + 1)
            )
            
            # Normalize histogram (L2 normalization)
            if self.config.normalize_histograms and hist.sum() > 0:
                hist = hist.astype(np.float32)
                hist = hist / np.linalg.norm(hist)
            
            histograms[i] = hist
        
        return histograms


In [17]:

# %% [classifier]
class Classifier:
    """Handles classification using KNN or SVM"""
    
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.model = None
        self.scaler = None
        self._create_model()
    
    def _create_model(self):
        """Create the appropriate classifier based on config"""
        if self.config.classifier_type.lower() == 'knn':
            self.model = KNeighborsClassifier(
                n_neighbors=self.config.knn_neighbors,
                metric=self.config.knn_metric,
                weights=self.config.knn_weights
            )
            print(f"Created KNN classifier (k={self.config.knn_neighbors}, "
                  f"metric={self.config.knn_metric}, weights={self.config.knn_weights})")
        
        elif self.config.classifier_type.lower() == 'svm':
            self.model = SVC(
                kernel=self.config.svm_kernel,
                C=self.config.svm_C,
                gamma=self.config.svm_gamma,
                degree=self.config.svm_degree,
                random_state=self.config.random_state
            )
            print(f"Created SVM classifier (kernel={self.config.svm_kernel}, "
                  f"C={self.config.svm_C}, gamma={self.config.svm_gamma})")
        
        else:
            raise ValueError(f"Unknown classifier type: {self.config.classifier_type}. "
                           "Use 'knn' or 'svm'")
        
        # Create scaler if needed
        if self.config.use_feature_scaling:
            self.scaler = StandardScaler()
    
    def train(self, X_train: np.ndarray, y_train: np.ndarray):
        """Train classifier with optional feature scaling"""
        print(f"Training {self.config.classifier_type.upper()} classifier...")
        
        # Apply feature scaling if enabled
        if self.scaler is not None:
            print("Applying StandardScaler to features...")
            X_train = self.scaler.fit_transform(X_train)
        
        self.model.fit(X_train, y_train)
        print("Training complete!")
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Make predictions with optional scaling"""
        if self.scaler is not None:
            X = self.scaler.transform(X)
        return self.model.predict(X)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Get prediction probabilities (if supported)"""
        if self.scaler is not None:
            X = self.scaler.transform(X)
        
        # KNN supports predict_proba
        if hasattr(self.model, 'predict_proba'):
            return self.model.predict_proba(X)
        # SVM with probability=True would support it, but we didn't enable it
        else:
            print("Warning: predict_proba not available for this classifier")
            return None
    
    def evaluate(self, X_test: np.ndarray, y_test: np.ndarray, 
                 class_names: List[str]) -> Dict[str, Any]:
        """
        Evaluate model and return metrics
        
        Returns:
            metrics: Dictionary containing various performance metrics
        """
        y_pred = self.predict(X_test)
        
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
            'recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
            'f1_score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'confusion_matrix': confusion_matrix(y_test, y_pred),
            'classification_report': classification_report(
                y_test, y_pred, 
                target_names=class_names,
                output_dict=True,
                zero_division=0
            )
        }
        
        return metrics

# %% [experiment_runner]
class ExperimentRunner:
    """Runs complete experiment pipeline"""
    
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.results = {}
        
    def run(self) -> Dict[str, Any]:
        """Execute complete experiment pipeline"""
        start_time = datetime.now()
        print(f"\n{'='*60}")
        print(f"Running Experiment: {self.config.experiment_name}")
        print(f"{'='*60}\n")
        
        # 1. Load data
        data_loader = DataLoader(self.config)
        images, labels, class_names = data_loader.load_images_and_labels()
        
        # 2. Extract features
        feature_extractor = FeatureExtractor(self.config)
        image_descriptors, all_descriptors = feature_extractor.extract_descriptors(images)
        
        # 3. Build vocabulary
        vocab_builder = VocabularyBuilder(self.config)
        vocabulary = vocab_builder.build_vocabulary(all_descriptors)
        
        # 4. Compute histograms
        histograms = vocab_builder.compute_histograms(image_descriptors)
        
        # 5. Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            histograms, labels,
            test_size=self.config.test_size,
            random_state=self.config.random_state,
            stratify=labels
        )
        
        print(f"Train set: {X_train.shape[0]} samples")
        print(f"Test set: {X_test.shape[0]} samples")
        
        # 6. Train classifier
        classifier = Classifier(self.config)
        classifier.train(X_train, y_train)
        
        # 7. Evaluate
        metrics = classifier.evaluate(X_test, y_test, class_names)
        
        # Store results
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        self.results = {
            'config': self.config.to_dict(),
            'metrics': metrics,
            'class_names': class_names,
            'timestamp': start_time.isoformat(),
            'duration_seconds': duration,
            'data_stats': {
                'n_images': len(images),
                'n_classes': len(class_names),
                'n_train': X_train.shape[0],
                'n_test': X_test.shape[0],
                'n_descriptors': all_descriptors.shape[0]
            }
        }
        
        # Print summary
        self._print_summary()
        
        return self.results
    
    def _print_summary(self):
        """Print experiment summary"""
        print(f"\n{'='*60}")
        print("EXPERIMENT RESULTS")
        print(f"{'='*60}")
        print(f"Experiment: {self.config.experiment_name}")
        print(f"Duration: {self.results['duration_seconds']:.2f} seconds")
        print(f"\nMetrics:")
        print(f"  Accuracy:  {self.results['metrics']['accuracy']:.4f}")
        print(f"  Precision: {self.results['metrics']['precision']:.4f}")
        print(f"  Recall:    {self.results['metrics']['recall']:.4f}")
        print(f"  F1-Score:  {self.results['metrics']['f1_score']:.4f}")
        print(f"{'='*60}\n")

# %% [hyperparameter_tuning]
class HyperparameterTuner:
    """Performs hyperparameter tuning using grid search"""
    
    def __init__(self, base_config: ExperimentConfig):
        self.base_config = base_config
        self.results = []
    
    def tune(self, param_grid: Dict[str, List]) -> pd.DataFrame:
        """
        Run grid search over parameter combinations
        
        Args:
            param_grid: Dictionary mapping parameter names to lists of values
                Example: {
                    'vocab_size': [50, 100, 150],
                    'knn_neighbors': [3, 5, 7],
                }
        
        Returns:
            results_df: DataFrame with all experiment results
        """
        from itertools import product
        
        # Generate all combinations
        param_names = list(param_grid.keys())
        param_values = list(param_grid.values())
        combinations = list(product(*param_values))
        
        print(f"Starting grid search with {len(combinations)} combinations...")
        print(f"Parameters: {param_names}\n")
        
        for i, param_combo in enumerate(combinations, 1):
            # Create config for this combination
            config = ExperimentConfig(**asdict(self.base_config))
            
            # Update parameters
            for param_name, param_value in zip(param_names, param_combo):
                setattr(config, param_name, param_value)
            
            # Set experiment name
            config.experiment_name = f"exp_{i:03d}_" + "_".join(
                f"{name}={value}" for name, value in zip(param_names, param_combo)
            )
            
            print(f"\n[{i}/{len(combinations)}] {config.experiment_name}")
            
            # Run experiment
            runner = ExperimentRunner(config)
            results = runner.run()
            self.results.append(results)
        
        # Convert to DataFrame
        results_df = self._results_to_dataframe()
        return results_df
    
    def _results_to_dataframe(self) -> pd.DataFrame:
        """Convert results to pandas DataFrame"""
        records = []
        
        for result in self.results:
            record = {
                'experiment_name': result['config']['experiment_name'],
                'timestamp': result['timestamp'],
                'duration_seconds': result['duration_seconds'],
                
                # Configuration
                'classifier_type': result['config']['classifier_type'],
                'vocab_size': result['config']['vocab_size'],
                'image_max_size': result['config']['image_max_size'],
                'normalize_histograms': result['config']['normalize_histograms'],
                'use_feature_scaling': result['config']['use_feature_scaling'],
                'use_minibatch_kmeans': result['config']['use_minibatch_kmeans'],
                
                # KNN parameters
                'knn_neighbors': result['config']['knn_neighbors'],
                'knn_metric': result['config']['knn_metric'],
                'knn_weights': result['config']['knn_weights'],
                
                # SVM parameters
                'svm_kernel': result['config']['svm_kernel'],
                'svm_C': result['config']['svm_C'],
                'svm_gamma': result['config']['svm_gamma'],
                
                # Metrics
                'accuracy': result['metrics']['accuracy'],
                'precision': result['metrics']['precision'],
                'recall': result['metrics']['recall'],
                'f1_score': result['metrics']['f1_score'],
                
                # Data stats
                'n_images': result['data_stats']['n_images'],
                'n_descriptors': result['data_stats']['n_descriptors'],
            }
            records.append(record)
        
        return pd.DataFrame(records)


In [18]:


# %% [results_visualizer]
class ResultsVisualizer:
    """Visualizes experiment results"""
    
    @staticmethod
    def plot_confusion_matrix(metrics: Dict, class_names: List[str], 
                             experiment_name: str = "", figsize=(10, 8)):
        """Plot confusion matrix heatmap"""
        cm = metrics['confusion_matrix']
        
        plt.figure(figsize=figsize)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names,
                   yticklabels=class_names)
        
        plt.title(f'Confusion Matrix\n{experiment_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_metrics_comparison(results_df: pd.DataFrame, figsize=(12, 6)):
        """Plot comparison of metrics across experiments"""
        metrics = ['accuracy', 'precision', 'recall', 'f1_score']
        
        fig, axes = plt.subplots(2, 2, figsize=figsize)
        axes = axes.ravel()
        
        for i, metric in enumerate(metrics):
            ax = axes[i]
            results_df.plot(x='experiment_name', y=metric, 
                          kind='bar', ax=ax, legend=False)
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_xlabel('')
            ax.set_ylabel('Score')
            ax.tick_params(axis='x', rotation=45)
            ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_parameter_impact(results_df: pd.DataFrame, param_name: str, 
                             metric: str = 'accuracy', figsize=(10, 6)):
        """Plot impact of a specific parameter on performance"""
        plt.figure(figsize=figsize)
        
        grouped = results_df.groupby(param_name)[metric].agg(['mean', 'std'])
        
        plt.errorbar(grouped.index, grouped['mean'], 
                    yerr=grouped['std'], marker='o', capsize=5)
        
        plt.title(f'Impact of {param_name} on {metric}')
        plt.xlabel(param_name)
        plt.ylabel(metric.replace('_', ' ').title())
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_best_vs_worst(results_df: pd.DataFrame, metric: str = 'accuracy',
                          n_experiments: int = 5, figsize=(12, 6)):
        """Compare best and worst performing experiments"""
        sorted_df = results_df.sort_values(metric, ascending=False)
        
        best_df = sorted_df.head(n_experiments)
        worst_df = sorted_df.tail(n_experiments)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        
        # Best experiments
        best_df.plot(x='experiment_name', y=metric, kind='barh', 
                    ax=ax1, legend=False, color='green')
        ax1.set_title(f'Top {n_experiments} Experiments')
        ax1.set_xlabel(metric.title())
        
        # Worst experiments
        worst_df.plot(x='experiment_name', y=metric, kind='barh', 
                     ax=ax2, legend=False, color='red')
        ax2.set_title(f'Bottom {n_experiments} Experiments')
        ax2.set_xlabel(metric.title())
        
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_classifier_comparison(results_df: pd.DataFrame, figsize=(14, 5)):
        """Compare KNN vs SVM performance across different metrics"""
        metrics = ['accuracy', 'precision', 'recall', 'f1_score']
        
        # Group by classifier type
        classifier_stats = results_df.groupby('classifier_type')[metrics].agg(['mean', 'std'])
        
        fig, axes = plt.subplots(1, 4, figsize=figsize)
        
        for i, metric in enumerate(metrics):
            ax = axes[i]
            
            # Get mean and std for each classifier
            means = classifier_stats[metric]['mean']
            stds = classifier_stats[metric]['std']
            
            x = np.arange(len(means))
            ax.bar(x, means, yerr=stds, capsize=5, alpha=0.7,
                  color=['#3498db', '#e74c3c'])
            ax.set_xticks(x)
            ax.set_xticklabels(means.index, rotation=0)
            ax.set_ylabel('Score')
            ax.set_title(metric.replace('_', ' ').title())
            ax.set_ylim([0, 1])
            ax.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.suptitle('KNN vs SVM - Performance Comparison', y=1.02, fontsize=14, fontweight='bold')
        plt.show()
    
    @staticmethod
    def plot_svm_kernel_comparison(results_df: pd.DataFrame, figsize=(10, 6)):
        """Compare different SVM kernels"""
        svm_results = results_df[results_df['classifier_type'] == 'svm']
        
        if len(svm_results) == 0:
            print("No SVM results found in DataFrame")
            return
        
        kernel_stats = svm_results.groupby('svm_kernel').agg({
            'accuracy': ['mean', 'std', 'count'],
            'f1_score': ['mean', 'std']
        })
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
        
        # Accuracy comparison
        kernels = kernel_stats.index
        acc_means = kernel_stats['accuracy']['mean']
        acc_stds = kernel_stats['accuracy']['std']
        
        ax1.bar(kernels, acc_means, yerr=acc_stds, capsize=5, alpha=0.7)
        ax1.set_ylabel('Accuracy')
        ax1.set_title('SVM Kernel Comparison - Accuracy')
        ax1.grid(axis='y', alpha=0.3)
        
        # F1-Score comparison
        f1_means = kernel_stats['f1_score']['mean']
        f1_stds = kernel_stats['f1_score']['std']
        
        ax2.bar(kernels, f1_means, yerr=f1_stds, capsize=5, alpha=0.7, color='orange')
        ax2.set_ylabel('F1-Score')
        ax2.set_title('SVM Kernel Comparison - F1-Score')
        ax2.grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()

# %% [experiment_manager]
class ExperimentManager:
    """Manages saving and loading experiment results"""
    
    def __init__(self, output_dir: str = "experiments"):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
    
    def save_results(self, results: Dict, filename: str = None):
        """Save experiment results to JSON"""
        if filename is None:
            filename = f"{results['config']['experiment_name']}.json"
        
        filepath = self.output_dir / filename
        
        # Convert numpy arrays to lists for JSON serialization
        results_serializable = self._make_serializable(results)
        
        with open(filepath, 'w') as f:
            json.dump(results_serializable, f, indent=2)
        
        print(f"Results saved to: {filepath}")
    
    def save_results_dataframe(self, df: pd.DataFrame, filename: str = "all_results.csv"):
        """Save results DataFrame to CSV"""
        filepath = self.output_dir / filename
        df.to_csv(filepath, index=False)
        print(f"Results DataFrame saved to: {filepath}")
    
    def load_results(self, filename: str) -> Dict:
        """Load experiment results from JSON"""
        filepath = self.output_dir / filename
        
        with open(filepath, 'r') as f:
            results = json.load(f)
        
        return results
    
    def _make_serializable(self, obj):
        """Convert numpy arrays and other non-serializable objects"""
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: self._make_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._make_serializable(item) for item in obj]
        elif isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        else:
            return obj




In [19]:

if __name__ == "__main__":
    # Define base configuration
    base_config = ExperimentConfig(
        data_dir="flowers",
        experiment_description="Comprehensive grid search: KNN vs SVM"
    )
    
    # Define parameter grid - comparing both classifiers
    param_grid = {
        'vocab_size': [50, 100, 150],
        'classifier_type': ['knn', 'svm'],
        'knn_neighbors': [3, 5, 7],  # Only affects KNN experiments
        'svm_kernel': ['linear', 'rbf'],  # Only affects SVM experiments
        'svm_C': [0.1, 1.0, 10.0],  # Only affects SVM experiments
        'use_feature_scaling': [True, False]
    }
    
    # Run grid search
    tuner = HyperparameterTuner(base_config)
    results_df = tuner.tune(param_grid)
    
    # Display results
    print("\n" + "="*60)
    print("GRID SEARCH RESULTS - TOP 10 CONFIGURATIONS")
    print("="*60)
    top_results = results_df.sort_values('accuracy', ascending=False).head(10)
    print(top_results[['experiment_name', 'classifier_type', 'vocab_size', 
                       'accuracy', 'f1_score', 'duration_seconds']])
    
    # Compare KNN vs SVM
    print("\n" + "="*60)
    print("KNN vs SVM COMPARISON")
    print("="*60)
    comparison = results_df.groupby('classifier_type').agg({
        'accuracy': ['mean', 'std', 'max'],
        'f1_score': ['mean', 'std', 'max'],
        'duration_seconds': ['mean', 'std']
    }).round(4)
    print(comparison)
    
    # Visualize
    visualizer = ResultsVisualizer()
    visualizer.plot_metrics_comparison(results_df.head(20))
    visualizer.plot_parameter_impact(results_df, 'vocab_size')
    
    # Compare classifiers
    visualizer.plot_classifier_comparison(results_df)
    
    # Best of each type
    best_knn = results_df[results_df['classifier_type'] == 'knn'].sort_values('accuracy', ascending=False).iloc[0]
    best_svm = results_df[results_df['classifier_type'] == 'svm'].sort_values('accuracy', ascending=False).iloc[0]
    
    print("\n" + "="*60)
    print("BEST KNN CONFIGURATION")
    print("="*60)
    print(f"Accuracy: {best_knn['accuracy']:.4f}")
    print(f"Vocab Size: {best_knn['vocab_size']}")
    print(f"K Neighbors: {best_knn['knn_neighbors']}")
    print(f"Metric: {best_knn['knn_metric']}")
    
    print("\n" + "="*60)
    print("BEST SVM CONFIGURATION")
    print("="*60)
    print(f"Accuracy: {best_svm['accuracy']:.4f}")
    print(f"Vocab Size: {best_svm['vocab_size']}")
    print(f"Kernel: {best_svm['svm_kernel']}")
    print(f"C: {best_svm['svm_C']}")
    
    # Save results
    manager = ExperimentManager()
    manager.save_results_dataframe(results_df)
    
    # Get best overall configuration
    best_idx = results_df['accuracy'].idxmax()
    best_config = results_df.loc[best_idx]
    print("\n" + "="*60)
    print("BEST OVERALL CONFIGURATION")
    print("="*60)
    print(best_config)

Starting grid search with 216 combinations...
Parameters: ['vocab_size', 'classifier_type', 'knn_neighbors', 'svm_kernel', 'svm_C', 'use_feature_scaling']


[1/216] exp_001_vocab_size=50_classifier_type=knn_knn_neighbors=3_svm_kernel=linear_svm_C=0.1_use_feature_scaling=True

Running Experiment: exp_001_vocab_size=50_classifier_type=knn_knn_neighbors=3_svm_kernel=linear_svm_C=0.1_use_feature_scaling=True

Loading images from: flowers
Found classes: ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']
Loaded 4317 images from 5 classes
File 'all_descriptors.npy' not found. Extracting descriptors...
Extracting SIFT descriptors...
  Processed 100/4317 images
  Processed 200/4317 images
  Processed 300/4317 images
  Processed 400/4317 images
  Processed 500/4317 images
  Processed 600/4317 images
  Processed 700/4317 images
  Processed 800/4317 images
  Processed 900/4317 images
  Processed 1000/4317 images
  Processed 1100/4317 images
  Processed 1200/4317 images
  Processed 1300/4317 imag

KeyboardInterrupt: 

In [ ]:

base_config = ExperimentConfig(
data_dir="UCMerced_LandUse",
experiment_description="Comprehensive grid search: KNN vs SVM"
)

# Define parameter grid - comparing both classifiers
param_grid = {
'vocab_size': [50, 100, 150,200],
'classifier_type': ['knn', 'svm'],
'knn_neighbors': [3, 5, 7],  # Only affects KNN experiments
'svm_kernel': ['linear', 'rbf'],  # Only affects SVM experiments
'svm_C': [0.1, 1.0, 10.0],  # Only affects SVM experiments
'use_feature_scaling': [True, False]
}

# Run grid search
tuner = HyperparameterTuner(base_config)
results_df = tuner.tune(param_grid)

# Display results
print("\n" + "="*60)
print("GRID SEARCH RESULTS - TOP 10 CONFIGURATIONS")
print("="*60)
top_results = results_df.sort_values('accuracy', ascending=False).head(10)
print(top_results[['experiment_name', 'classifier_type', 'vocab_size', 
   'accuracy', 'f1_score', 'duration_seconds']])

# Compare KNN vs SVM
print("\n" + "="*60)
print("KNN vs SVM COMPARISON")
print("="*60)
comparison = results_df.groupby('classifier_type').agg({
'accuracy': ['mean', 'std', 'max'],
'f1_score': ['mean', 'std', 'max'],
'duration_seconds': ['mean', 'std']
}).round(4)
print(comparison)

# Visualize
visualizer = ResultsVisualizer()
visualizer.plot_metrics_comparison(results_df.head(20))
visualizer.plot_parameter_impact(results_df, 'vocab_size')

# Compare classifiers
visualizer.plot_classifier_comparison(results_df)

# Best of each type
best_knn = results_df[results_df['classifier_type'] == 'knn'].sort_values('accuracy', ascending=False).iloc[0]
best_svm = results_df[results_df['classifier_type'] == 'svm'].sort_values('accuracy', ascending=False).iloc[0]

print("\n" + "="*60)
print("BEST KNN CONFIGURATION")
print("="*60)
print(f"Accuracy: {best_knn['accuracy']:.4f}")
print(f"Vocab Size: {best_knn['vocab_size']}")
print(f"K Neighbors: {best_knn['knn_neighbors']}")
print(f"Metric: {best_knn['knn_metric']}")

print("\n" + "="*60)
print("BEST SVM CONFIGURATION")
print("="*60)
print(f"Accuracy: {best_svm['accuracy']:.4f}")
print(f"Vocab Size: {best_svm['vocab_size']}")
print(f"Kernel: {best_svm['svm_kernel']}")
print(f"C: {best_svm['svm_C']}")

# Save results
manager = ExperimentManager()
manager.save_results_dataframe(results_df)

# Get best overall configuration
best_idx = results_df['accuracy'].idxmax()
best_config = results_df.loc[best_idx]
print("\n" + "="*60)
print("BEST OVERALL CONFIGURATION")
print("="*60)
print(best_config)

Starting grid search with 48 combinations...
Parameters: ['vocab_size', 'classifier_type', 'knn_neighbors', 'svm_kernel', 'svm_C', 'use_feature_scaling']


[1/48] exp_001_vocab_size=50_classifier_type=knn_knn_neighbors=5_svm_kernel=linear_svm_C=0.1_use_feature_scaling=True

Running Experiment: exp_001_vocab_size=50_classifier_type=knn_knn_neighbors=5_svm_kernel=linear_svm_C=0.1_use_feature_scaling=True

Loading images from: UCMerced_LandUse
Found classes: ['UCMerced_images_feats.npy', 'UCMerced_labels_num.npy', 'agricultural', 'airplane', 'baseballdiamond', 'beach', 'buildings', 'chaparral', 'denseresidential', 'forest', 'freeway', 'golfcourse', 'harbor', 'intersection', 'mediumresidential', 'mobilehomepark', 'overpass', 'parkinglot', 'river', 'runway', 'sparseresidential', 'storagetanks', 'tenniscourt']
Loaded 2100 images from 23 classes
Loaded all descriptors from 'all_descriptors.npy'
Building vocabulary with 50 visual words...
Vocabulary built. Shape: (50, 128)
Computing BoVW histo

ValueError: Found input variables with inconsistent numbers of samples: [4317, 2100]